# 16. LangGraph 개념 — StateGraph · Node · Edge · 조건부 분기
> Day 3 · 19H · 소요 약 50분

## 학습 목표

- **LCEL 체인의 한계 (DAG)** 를 이해하고 LangGraph 가 필요한 순간을 식별한다.
- `StateGraph`, `TypedDict` 기반 상태, `add_node` / `add_edge` / `add_conditional_edges` 를 손으로 조립한다.
- **조건부 분기 + 루프** 패턴 — "실패하면 이전 노드로 되돌아가 재시도" 를 구현한다.
- `stream()` 으로 각 노드의 실행 궤적을 추적한다.
- (맛보기) `create_react_agent` 로 **ReAct 에이전트** 한 줄 생성 — 17 번 SQL 에이전트의 동기부여.

> 본 노트북은 의도적으로 **SQL 을 쓰지 않습니다.** 17 번에서 LangGraph 를 실제 SQL 에이전트에 조립합니다. 여기서는 **그래프 문법 그 자체**에 집중합니다.

In [ ]:
%pip install -q langgraph langchain langchain-openai langchain-core

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다.
import os

def _load_secret(key: str, required: bool = True) -> None:
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")

_load_secret("OPENAI_API_KEY", required=True)
print("Environment ready.")

## 1. 왜 그래프인가 — LCEL vs LangGraph

```
LCEL 체인 (DAG, 단방향):
  A → B → C → D
  ❌ B 가 실패하면 A 로 되돌아갈 수 없음
  ❌ 조건에 따라 C 를 건너뛰는 분기 불가

LangGraph (FSM, 상태 머신):
  A → B → C → D
       ↑      ↓
       └──────┘   ← 실패 시 되돌아가서 재시도
  ✅ 루프·조건 분기·재시도 모두 가능
```

| 특성 | LCEL 체인 | LangGraph |
|---|---|---|
| 구조 | DAG | FSM (상태 머신) |
| 데이터 흐름 | 한 방향 파이프 | 공유 상태(State) 읽기/쓰기 |
| 에러 처리 | fallback (대체 모델) | 재시도 루프, 조건 분기 |
| 복잡도 | 단순 | 중간 |
| 적합한 경우 | 단순 RAG, 변환 | 에이전트, 멀티스텝, 재시도 |

**한 줄 요약:** LangGraph = "상태(State) 를 중심으로 도는 그래프 실행 엔진".

## 2. 최소 예제 — 3개 노드 순차 실행

가장 간단한 그래프: `greet → process → finish`. `SimpleState` 라는 공유 상태에 `message` 와 `step` 을 기록하면서 지나갑니다.

핵심 문법:
- `TypedDict` 로 상태 스키마 선언.
- 각 노드는 `(State) -> dict` 시그니처. 반환 dict 는 **변경할 키만** 담으면 됩니다 (부분 업데이트).
- `set_entry_point(...)` 로 시작 노드 지정, `END` 로 종료.

In [ ]:
# LangGraph 의 가장 작은 예제 — 노드 3 개를 직선으로 연결.
# TypedDict 로 "이 그래프가 다루는 상태(State) 의 형태" 를 미리 선언합니다.
from typing import TypedDict
from langgraph.graph import StateGraph, END


class SimpleState(TypedDict):
    message: str   # 노드들이 메시지를 점점 누적해 갈 필드
    step: int      # 단순 카운터 — 몇 번째 단계인지 추적용


# 모든 노드 함수의 약속:
#   - 입력: 현재 상태(SimpleState dict)
#   - 출력: dict (상태에 덮어쓸 키들만 담음 — 전체를 다시 만들 필요 없음)
def greet(state: SimpleState) -> dict:
    return {
        "message": f"안녕하세요! (step={state['step']})",
        "step": state["step"] + 1,
    }


def process(state: SimpleState) -> dict:
    return {
        "message": state["message"] + " → 처리 완료!",
        "step": state["step"] + 1,
    }


def finish(state: SimpleState) -> dict:
    return {
        "message": state["message"] + " → 종료.",
        "step": state["step"] + 1,
    }


# 그래프 조립 — 빌드 단계와 실행 단계가 분리되어 있다는 것이 LangGraph 의 핵심.
graph = StateGraph(SimpleState)        # ① "어떤 상태를 다룰지" 선언
graph.add_node("greet", greet)         # ② 노드(=실행할 함수) 등록 (이름은 자유)
graph.add_node("process", process)
graph.add_node("finish", finish)

graph.set_entry_point("greet")          # ③ 시작 노드 지정
graph.add_edge("greet", "process")      # ④ 직선 엣지 — A 끝나면 B 로
graph.add_edge("process", "finish")
graph.add_edge("finish", END)            # END = "여기서 그래프 종료"

# ⑤ .compile() 은 빌더에서 "실제 실행 가능한 객체" 를 만들어 반환.
app = graph.compile()

# .invoke() 는 그래프 끝까지 실행 후 최종 상태를 dict 로 반환.
result = app.invoke({"message": "", "step": 0})
print(f"최종 메시지: {result['message']}")
print(f"총 스텝: {result['step']}")

## 3. 그래프 시각화

`get_graph().draw_mermaid_png()` 가 Colab 환경에서는 작동하지만, 네트워크/그래프비즈 설정에 따라 실패할 수 있으니 `try/except` 로 감싸고 fallback 으로 **Mermaid 텍스트** 를 출력합니다.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"[info] mermaid png 렌더 실패 ({e}) → 텍스트로 출력:")
    print(app.get_graph().draw_mermaid())

## 4. 조건부 분기 + 루프 — 재시도 패턴

20H SQL 에이전트의 핵심 모티프: **실패하면 이전 노드로 되돌아간다.** LCEL 로는 불가능한 패턴입니다.

- `add_conditional_edges(source, router_fn, path_map)` — `router_fn(state)` 이 반환하는 라벨을 `path_map` 으로 **실제 노드 이름** 에 매핑.
- 이 예시는 30% 확률로 실패하는 `try_task` 를 최대 3 회까지 재시도합니다.

In [ ]:
# 조건부 분기 + 루프 = LangGraph 의 진짜 가치.
# 같은 노드에서 다음 행선지가 "상태 값에 따라" 동적으로 바뀝니다 — LCEL 로는 표현 불가능.
import random
from typing import TypedDict


class RetryState(TypedDict):
    task: str       # 작업 이름 (디버깅용)
    result: str     # 성공 시 결과 메시지
    error: str      # 실패 시 에러 메시지 (이 값이 라우팅의 키)
    attempts: int   # 시도 횟수 — 무한 루프 방지에 필수


def try_task(state: RetryState) -> dict:
    """30% 확률로 실패하는 가상 작업. 실제 SQL 에이전트의 execute_sql 노드와 같은 자리."""
    attempts = state.get("attempts", 0) + 1
    # random.random() 은 0.0~1.0 사이 실수. 0.3 미만이면 실패 분기로.
    if random.random() < 0.3 and attempts <= 3:
        return {
            "error": f"시도 {attempts}에서 랜덤 에러 발생!",
            "attempts": attempts,
        }
    # 성공 시 error 를 빈 문자열로 비워 다음 라우터가 'success' 를 반환하게 만든다.
    return {
        "result": f"✅ 시도 {attempts}에서 성공!",
        "error": "",
        "attempts": attempts,
    }


def handle_success(state: RetryState) -> dict:
    return {"result": state["result"] + " → 완료 처리됨."}


def should_retry(state: RetryState) -> str:
    """라우팅 함수 — 반환 문자열이 add_conditional_edges 의 path_map 키와 일치해야 한다."""
    if not state.get("error"):
        return "success"             # 에러 없음 → 성공 노드로
    if state.get("attempts", 0) >= 3:
        return "success"             # 3회 도달 → 강제 종료 (재시도 포기)
    return "retry"                   # 그 외 → 다시 try_task 로 (셀프 루프)


retry_graph = StateGraph(RetryState)
retry_graph.add_node("try_task", try_task)
retry_graph.add_node("success", handle_success)

retry_graph.set_entry_point("try_task")
# 핵심 포인트: try_task 의 "다음 노드"가 should_retry 의 반환값에 따라 결정됨.
# path_map 의 키("success"/"retry") = should_retry 가 반환하는 문자열,
# 값("success"/"try_task") = 실제 갈 노드 이름.
retry_graph.add_conditional_edges(
    "try_task",
    should_retry,
    {"success": "success", "retry": "try_task"},  # retry → 자기 자신으로 되돌아오는 self-loop
)
retry_graph.add_edge("success", END)

retry_app = retry_graph.compile()

random.seed(42)  # 시드 고정 — 매번 같은 결과가 나와 학생 간 비교가 쉬움
for i in range(5):
    result = retry_app.invoke(
        {"task": "데이터 처리", "result": "", "error": "", "attempts": 0}
    )
    print(f"[실행 {i+1}] 총 시도 {result['attempts']}회: {result['result']}")

### 재시도 그래프 시각화

`try_task` 노드에서 자기 자신으로 되돌아가는 **self-loop** 엣지가 그려집니다.

In [ ]:
try:
    display(Image(retry_app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"[info] mermaid png 렌더 실패 ({e}) → 텍스트로 출력:")
    print(retry_app.get_graph().draw_mermaid())

## 5. stream() 으로 실행 궤적 추적

`invoke()` 는 최종 결과만 반환합니다. 디버깅할 때는 `stream()` 으로 **각 노드가 실행될 때마다 이벤트** 를 받아야 합니다. 17 번 SQL 에이전트에서도 `for event in agent.stream(...)` 패턴을 그대로 씁니다.

In [ ]:
random.seed(7)  # 이 시드는 2회 재시도 후 성공하도록 설정

print("🔍 실행 궤적:")
for event in retry_app.stream(
    {"task": "추적 테스트", "result": "", "error": "", "attempts": 0}
):
    for node_name, node_output in event.items():
        attempts = node_output.get("attempts", "?")
        error = (node_output.get("error") or "")[:30]
        result = (node_output.get("result") or "")[:30]
        print(f"  📍 {node_name:12s} | attempts={attempts} | error='{error}' | result='{result}'")

### stream() vs invoke() 정리

- `invoke(state)` — 최종 상태만 반환. 프로덕션 호출용.
- `stream(state)` — 각 노드 실행 직후 **상태 업데이트 이벤트** 를 yield. 디버깅·UI 스트리밍용.
- 이벤트 구조: `{<node_name>: <partial_update_dict>}`.

## 6. 분류 → 분기 → 응답 — 3 노드 조건 그래프

실전적인 미니 예제: 사용자 질문을 **LLM 으로 분류** 한 뒤 카테고리별로 다른 응답 노드로 분기합니다.

- `classify` — 질문을 `greeting` / `question` / `other` 로 분류.
- `answer_greeting` / `answer_question` / `answer_other` — 카테고리별 응답.

이 패턴은 실제 고객 응대 챗봇·FAQ 라우터에서 흔히 쓰이는 구조입니다.

In [ ]:
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


class ChatState(TypedDict):
    question: str
    category: str
    answer: str


classify_prompt = ChatPromptTemplate.from_template(
    "다음 사용자 입력을 카테고리 하나로만 분류하세요.\n"
    "선택지: greeting, question, other\n"
    "입력: {question}\n"
    "카테고리(소문자 한 단어만):"
)
classify_chain = classify_prompt | chat_llm | StrOutputParser()


def classify(state: ChatState) -> dict:
    raw = classify_chain.invoke({"question": state["question"]}).strip().lower()
    cat = raw if raw in {"greeting", "question", "other"} else "other"
    return {"category": cat}


def answer_greeting(state: ChatState) -> dict:
    return {"answer": "안녕하세요! 무엇을 도와드릴까요?"}


def answer_question(state: ChatState) -> dict:
    # 실제 질문 응답은 LLM 에 맡김 (간단 예시)
    ans = chat_llm.invoke(f"다음 질문에 한 문장으로 답하세요: {state['question']}").content
    return {"answer": ans.strip()}


def answer_other(state: ChatState) -> dict:
    return {"answer": "질문 또는 인사 외의 입력은 지원하지 않습니다."}


def route_by_category(state: ChatState) -> str:
    return state["category"]  # greeting / question / other


chat_graph = StateGraph(ChatState)
chat_graph.add_node("classify", classify)
chat_graph.add_node("answer_greeting", answer_greeting)
chat_graph.add_node("answer_question", answer_question)
chat_graph.add_node("answer_other", answer_other)

chat_graph.set_entry_point("classify")
chat_graph.add_conditional_edges(
    "classify",
    route_by_category,
    {
        "greeting": "answer_greeting",
        "question": "answer_question",
        "other":    "answer_other",
    },
)
for n in ("answer_greeting", "answer_question", "answer_other"):
    chat_graph.add_edge(n, END)

chat_app = chat_graph.compile()

for q in ["안녕!", "심장내과는 어떤 과야?", "피자 주문해줘"]:
    r = chat_app.invoke({"question": q, "category": "", "answer": ""})
    print(f"❓ {q}\n   → [{r['category']}] {r['answer']}\n")

## 7. (맛보기) ReAct 에이전트 — 한 줄 생성

LangGraph 는 자주 쓰는 에이전트 패턴을 프리빌트(prebuilt) 로 제공합니다. **ReAct (Reason + Act)** — LLM 이 "생각 → 도구 호출 → 관찰 → 다시 생각 …" 을 반복하는 전형적 에이전트 구조.

아래는 **계산기 도구** 하나만 붙인 최소 ReAct 에이전트 예시입니다. `create_react_agent(model, tools)` 한 줄로 LangGraph 그래프가 나오고, 내부는 4 번처럼 조건부 루프로 구성돼 있습니다.

> **주의:** 17 번은 이 prebuilt 를 쓰지 **않습니다.** SQL 에이전트는 교육 목적상 **StateGraph 를 직접 조립** 해 내부 구조를 이해하는 게 목표입니다. ReAct prebuilt 는 "이런 추상화도 있다" 정도로만 맛봅니다.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent


@tool
def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression like '3 * (4 + 5)'."""
    # NOTE: demo only. 실제 프로덕션에서는 eval 을 절대 쓰지 마세요.
    try:
        allowed = set("0123456789+-*/(). ")
        if not set(expression) <= allowed:
            return "ERR: 허용되지 않는 문자가 포함되었습니다."
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"ERR: {e}"


react_agent = create_react_agent(chat_llm, tools=[calculator])

react_result = react_agent.invoke(
    {"messages": [("user", "12 * (7 + 3) 은 얼마야? calculator 를 써서 답해줘.")]}
)
print("최종 응답:")
print(react_result["messages"][-1].content)

## 실습 과제

1. `SimpleState` 그래프에 `log` 노드를 추가하세요 — 실행된 메시지를 `log: list` 필드에 append 하고 마지막에 한꺼번에 출력. `process → log → finish` 순서가 되도록 엣지를 재배선.
2. `RetryState` 의 `should_retry` 에서 **최대 재시도 횟수를 5** 로 변경하고, 30% 실패 확률을 60% 로 올려 재시도가 실제로 5 회까지 가는 경우를 관찰하세요.
3. `ChatState` 라우터에 **`complaint`** (민원) 카테고리를 추가하세요. `classify` 프롬프트와 `path_map` 둘 다 수정해야 합니다. 라벨과 `path_map` 키가 **정확히 일치** 해야 한다는 점을 체감.

In [ ]:
# TODO 1: log 노드 추가
# from typing import TypedDict
# class LogState(TypedDict):
#     message: str
#     step: int
#     log: list
#
# def log_node(state: LogState) -> dict:
#     entry = f"[step {state['step']}] {state['message']}"
#     return {"log": state.get("log", []) + [entry]}
#
# g = StateGraph(LogState)
# ... add_node / add_edge 로 greet → process → log → finish 구성


# TODO 2: 재시도 한도 5 로 확장
# MAX_ATTEMPTS = 5
# def should_retry_v2(state: RetryState) -> str:
#     if not state.get("error"): return "success"
#     if state.get("attempts", 0) >= MAX_ATTEMPTS: return "success"
#     return "retry"


# TODO 3: complaint 카테고리 추가
# classify_prompt_v2 = ChatPromptTemplate.from_template(
#     "... 선택지: greeting, question, complaint, other ..."
# )
# path_map_v2 = {
#     "greeting":  "answer_greeting",
#     "question":  "answer_question",
#     "complaint": "answer_complaint",   # 신규 노드
#     "other":     "answer_other",
# }

## 다음 노트북에서는…

**`17_my_sql_agent.ipynb`** — 오늘 배운 `StateGraph` + 조건부 분기 + 재시도 패턴을 그대로 가져다 **SQL 에이전트** 를 조립합니다. 노드 4 개 (`generate_sql → run_sql → validate → answer`), 에러 시 재시도 루프, `stream()` 추적까지 — 과정 전체의 정점입니다. 과제 #3 의 기반이 되는 노트북이니 꼼꼼히 따라오세요.